# C6_01 - Agent RAG simplu pentru o bulă discursivă

În C5 am construit memoria semantică a unei bule: texte curate, embeddings, FAISS și metadate.
În C6 folosim această memorie pentru a genera primul răspuns RAG al agentului.
Fluxul este:
```text
input politic nou
→ regăsire semantică în FAISS
→ top-k fragmente relevante
→ rol din roles.yaml
→ șablon de prompt
→ LLM
→ răspuns al agentului


## 0. Setup și poziționare în proiect
Notebook-ul poate fi rulat din `notebooks/student_XX/`, dar fișierele proiectului sunt în rădăcina repository-ului.
De aceea, mai întâi ne asigurăm că lucrăm din folderul principal al proiectului.

In [1]:
from pathlib import Path
import os
import json
import pickle

import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

c:\Users\nasta\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.chdir(Path.cwd().parents[1])

print("Folder proiect:", Path.cwd())
print("data/bubbles:", Path("data/bubbles").exists())
print("assets/vectorstores:", Path("assets/vectorstores").exists())

Folder proiect: d:\ADC\AN 2\18. AI Algorithms\AI_Project\echochamber-project-team-6
data/bubbles: True
assets/vectorstores: True


În C5, fiecare bulă trebuie să aibă:
```text
data/bubbles/<agent_slug>.jsonl
assets/vectorstores/<agent_slug>/index.faiss
assets/vectorstores/<agent_slug>/index.pkl

## 1. Aleg agentul meu
Fiecare membru al echipei lucrează pe o singură bulă discursivă. Alegem agentul, apoi verificăm dacă există fișierele construite în C5 pentru acel agent.


- `MY_AGENT` este numele tehnic al bulei pe care o folosim.
- `K = 5`  sistemul va recupera primele 5 fragmente cele mai apropiate semantic de inputul nostru.


In [3]:
MY_AGENT = "pro_european"
K = 5

AGENTS = [
    "personalist_salvator",
    "anti_sistem",
    "anti_suveranist",
    "conspirationist",
    "pro_european",
]

assert MY_AGENT in AGENTS, f"Alege un agent valid: {AGENTS}"

bubble_path = Path("data/bubbles") / f"{MY_AGENT}.jsonl"
index_path = Path("assets/vectorstores") / MY_AGENT / "index.faiss"
metadata_path = Path("assets/vectorstores") / MY_AGENT / "index.pkl"

print("Agent ales:", MY_AGENT)
print("Bubble JSONL:", bubble_path.exists(), bubble_path)
print("FAISS index:", index_path.exists(), index_path)
print("Metadata:", metadata_path.exists(), metadata_path)

Agent ales: pro_european
Bubble JSONL: True data\bubbles\pro_european.jsonl
FAISS index: True assets\vectorstores\pro_european\index.faiss
Metadata: True assets\vectorstores\pro_european\index.pkl


## 2. Încarc rolul meu din `role_XX.yaml`
În C5, agentul era doar o categorie de corpus: un fișier `.jsonl` și un index FAISS.
În C6, agentul începe să răspundă. Pentru asta are nevoie de o voce, o poziție discursivă și reguli.
Fiecare membru al echipei lucrează într-un fișier separat:
```text
assets/roles/role_XX.yaml


student_01 → assets/roles/role_01.yaml
student_02 → assets/roles/role_02.yaml



#exemplu de rol:
anti_sistem:
  name: "Anti-sistem"
  voice: "critic, suspicios, moralizator"
  worldview: "instituțiile sunt suspecte sau compromise"
  rules:
    - "folosește contextul recuperat"
    - "nu inventa informații care nu apar în context"
    - "răspunde în 4-6 fraze"

In [4]:
import yaml
ROLES_PATH = Path("assets/roles/role_05.yaml")
print("Role file există:", ROLES_PATH.exists())

Role file există: True


In [5]:
with open(ROLES_PATH, "r", encoding="utf-8") as f:
    role_file = yaml.safe_load(f)
role = role_file[MY_AGENT]

print("Agent:", role["name"])
print("Slug:", role["slug"])
print("Emoji:", role.get("emoji", ""))
print("Color:", role.get("color", ""))
print("\nSystem prompt:\n")
print(role["system"])

Agent: pro_european
Slug: pro_european
Emoji: ✧
Color: #e29c4a

System prompt:

Ești un un cetățean românn pro-european, care conștientizează avantajele apartenenței la Uniunea Europeană, care apără instituțiile UE.
Crezi că instituțiile UE sunt profund necesare, lupți pentru democrație, meritocrație și transparență.
Cum vorbești:
- echilibrat, entuziasmat, didactic
- cu claritate juridică, clar
- argumente contra extremismului, izolaționismului și manipulării
Ce te definește:
- nu ai încredere în extremism, izolaționism
- susții Uniunea Europeană și instituțiile sale
- susții statul de drept și ancorarea europeană
- faci acuza împotriva populismului
-ești pro-european în primul rand nu anti-suveranist
Vei primi:
[STIMULUS] — știrea sau textul la care reacționezi
[COMENTARII SIMILARE] — exemple reale din corpus, utile pentru ton și stil
Reguli:
- scrii ca un comentariu autentic de YouTube în limba română
- folosești comentariile similare doar ca inspirație de ton, nu le copia
- nu expl

Ce face codul:
- `ROLES_PATH` indică fișierul cu rolurile agenților.
- `yaml.safe_load()` citește fișierul YAML și îl transformă într-un dicționar Python.
- `roles[MY_AGENT]` selectează doar rolul agentului ales la pasul anterior.
- Afișăm numele, vocea, poziția discursivă și regulile, ca să verificăm dacă agentul este definit corect.
Verificare rapidă:
- vocea se potrivește cu bula aleasă?
- regulile cer folosirea contextului?
- regulile limitează inventarea informațiilor?

## 3. Încarc FAISS și metadatele din C5
În C5 am construit vectorstore-ul pentru fiecare bulă discursivă.
Acum reutilizăm acea muncă: încărcăm indexul FAISS și metadatele agentului ales.
```text
index.faiss = vectorii textelor
index.pkl   = textele originale și metadatele

In [6]:
index = faiss.read_index(str(index_path))

with open(metadata_path, "rb") as f:
    metadata = pickle.load(f)

print("Vectori în FAISS:", index.ntotal)
print("Texte în metadata:", len(metadata))
print("Dimensiune vectori:", index.d)

Vectori în FAISS: 50
Texte în metadata: 50
Dimensiune vectori: 384


In [7]:
metadata[0]

{'id': 'yt_yEuctxNb4O0_UgxTCkwcP96Sb5_Rpht4AaABAg',
 'text': 'Tipul care și-a dat demisia în noiembrie la cum gândește și vorbește merita sa fie ministrul justiției. Oameni tineri, capabili, încă necorupți, fără trecut securist/comunist trebuie sa fie în funcții cheie ale statului. Vezi miniștrii USR, maxim 40 de ani, pregătiți, implicați....',
 'source_channel': 'NicusorDanRO',
 'channel_family': 'mainstream_actor',
 'video_title': '🟢 LIVE - Întâlnire la Palatul Cotroceni cu magistrați și alți actori din sistemul judiciar',
 'target_refined': 'usr',
 'stance_to_target': 'pro',
 'confidence': 0.9,
 'discourse_type': 'T5_pro_democratic_european',
 'discourse_subtype': 'legitimitate_pluralista',
 'type_confidence': 'medium',
 'agent': 'Pro-european',
 'slug': 'pro_european',
 'personality': 'normativ, moderat, legalist',
 'speaks': 'sobru, justificativ, procedural',
 'definition': 'apără regulile, instituțiile și ancorarea europeană'}

In [8]:
assert index.ntotal == len(metadata), "Numărul de vectori nu corespunde cu numărul de texte din metadata."

print("Indexul FAISS și metadatele sunt aliniate.")

Indexul FAISS și metadatele sunt aliniate.


## 4. Recuperăm context pentru un input nou
Acum repetăm mecanismul din C5, dar îl folosim ca prim pas pentru generare.
Scriem un text politic nou, îl transformăm în reprezentare vectorială, apoi căutăm în FAISS fragmentele cele mai apropiate semantic.
Aceste fragmente vor deveni contextul pe care îl trimitem mai târziu către LLM.

In [9]:
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1485.80it/s]


In [10]:
input_text = "România trebuie să părăsească Uniunea Europeană."

query_embedding = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

scores, positions = index.search(query_embedding, K)

results = []

for score, pos in zip(scores[0], positions[0]):
    item = metadata[pos].copy()
    item["score"] = round(float(score), 3)
    results.append(item)

results_df = pd.DataFrame(results)

cols = [
    "score",
    "agent",
    "text",
    "source_channel",
    "video_title",
    "type_confidence",
    "discourse_subtype",
]

cols = [c for c in cols if c in results_df.columns]

results_df[cols]

,score,agent,text,source_channel,video_title,type_confidence,discourse_subtype
0,0.618,Pro-european,Visul Romaniei de la 1848 a fost sa faca polit...,NicusorDanRO,🟢 LIVE Declarații de presă susținute după part...,high,pro_european_ancorare
1,0.583,Pro-european,Unde este stegul Uniunii Europene si steagul N...,NicusorDanRO,🟢 LIVE Discursul susținut în cadrul recepției ...,high,pro_european_ancorare
2,0.580,Pro-european,"+ la asta, d. Președinte, trebuie sa existe O ...",NicusorDanRO,🟢 LIVE Declarații de presă înaintea participăr...,medium,legitimitate_pluralista
3,0.517,Pro-european,@Robert Turcescu Oficial: întrebare pentru dnu...,turcescu111,"TIC-TAC, TIC-TAC, pregătiți-vă: Călin Georgesc...",high,aparare_institutionala_procedurala
4,0.499,Pro-european,❤❤ NICUȘOR DAN președinte ♥️ MULȚUMIM UE Schen...,TuDecizi-s3g,Tu Decizi Live,high,pro_european_ancorare


Ce face codul:
- `input_text` este textul nou la care agentul va reacționa.
- `model.encode()` transformă textul într-o reprezentare vectorială.
- `normalize_embeddings=True` păstrează aceeași logică folosită în C5.
- `index.search(..., K)` caută primele `K` fragmente cele mai apropiate din FAISS.
- `metadata[pos]` recuperează textul original și metadatele corespunzătoare fiecărui vector.
- `score` arată cât de apropiat este fragmentul de inputul nostru.

### Verificare manuală
Citește cele 5 rezultate și notează câte sunt relevante pentru inputul tău.

In [11]:
relevant_results = 5  

print(f"Rezultate relevante: {relevant_results}/{K}")

Rezultate relevante: 5/5


Dacă rezultatele sunt slabe, problema poate veni din:
- input prea vag;
- bula aleasă nu conține texte potrivite;
- textele din `data/bubbles/<agent_slug>.jsonl` sunt prea puține sau prea generale;
- `K` este prea mic sau prea mare.

## 5. Construim contextul pentru LLM

LLM-ul nu primește tot corpusul. Primește doar fragmentele recuperate la pasul anterior.
Acum transformăm rezultatele FAISS într-un bloc de context clar, care poate fi introdus în prompt.
Păstrăm și scorurile/metadatele, ca să putem vedea de unde vine răspunsul.

In [12]:
context_parts = []

for i, item in enumerate(results, start=1):
    text = item.get("text", "")
    score = item.get("score", "")
    source = item.get("source_channel", "")
    title = item.get("video_title", "")
    
    context_parts.append(
        f"""[Fragment {i} | score={score} | source={source}]
{text}
"""
    )

retrieved_context = "\n".join(context_parts)

print(retrieved_context)

[Fragment 1 | score=0.618 | source=NicusorDanRO]
Visul Romaniei de la 1848 a fost sa faca politica Europeana. Sa fie inclusa in Europa si sa fie Europa. Avem acum acest lucru! Ne-am îndeplinit visul iar asta duce la o bunastare fantastica (Romania este cea mai prospera din istorie!) Si exista unii trepanati da ne spuna ca UE nu e nimic. Va dati seama!? Nu zic ca nu mai avem treaba. Mai avem enorm de mult. Dar avem si posibilitatea sa criticam guvernul. Ceea ce ex-EU nu permite. Acolo te “aliniezi” cu interesul national!

[Fragment 2 | score=0.583 | source=NicusorDanRO]
Unde este stegul Uniunii Europene si steagul NATO din imaginea de la Cotroceni?!...eu pt astea două steaguri împreună cu al României am votat 😠...să va fie ruşine România nu e nimeni si nimic fară 🇪🇺 & NATO ...

[Fragment 3 | score=0.58 | source=NicusorDanRO]
+ la asta, d. Președinte, trebuie sa existe O MAJORITATE a cetatenilor si in Romănia..., care sa doreasca UNIREA....

[Fragment 4 | score=0.517 | source=turcescu111

Ce face codul:
- ia cele `K` fragmente recuperate la pasul anterior;
- construiește un singur bloc de context;
- păstrează scorul și sursa fiecărui fragment;
- pregătește textul care va fi trimis către LLM.
Ideea importantă: contextul este o selecție. Modelul va răspunde doar pe baza fragmentelor pe care i le oferim.

In [13]:
print("Număr fragmente în context:", len(results))
print("Lungime context în caractere:", len(retrieved_context))

Număr fragmente în context: 5
Lungime context în caractere: 1729


## 6. RAG manual: construim promptul simplu
Înainte să folosim LangChain, construim promptul manual.
Scopul este să vedem clar cele trei piese ale agentului RAG:
1. rolul agentului;
2. textul nou la care reacționează;
3. contextul recuperat din FAISS.

In [14]:
agent_system = role["system"]

prompt = f"""
{agent_system}

[STIMULUS]
{input_text}

[COMENTARII SIMILARE]
{retrieved_context}
"""

print(prompt)


Ești un un cetățean românn pro-european, care conștientizează avantajele apartenenței la Uniunea Europeană, care apără instituțiile UE.
Crezi că instituțiile UE sunt profund necesare, lupți pentru democrație, meritocrație și transparență.
Cum vorbești:
- echilibrat, entuziasmat, didactic
- cu claritate juridică, clar
- argumente contra extremismului, izolaționismului și manipulării
Ce te definește:
- nu ai încredere în extremism, izolaționism
- susții Uniunea Europeană și instituțiile sale
- susții statul de drept și ancorarea europeană
- faci acuza împotriva populismului
-ești pro-european în primul rand nu anti-suveranist
Vei primi:
[STIMULUS] — știrea sau textul la care reacționezi
[COMENTARII SIMILARE] — exemple reale din corpus, utile pentru ton și stil
Reguli:
- scrii ca un comentariu autentic de YouTube în limba română
- folosești comentariile similare doar ca inspirație de ton, nu le copia
- nu explica ce faci
- nu face liste
- nu folosi ghilimele
- răspunde cu un singur comen

Ce face codul:
- `agent_system` ia rolul agentului din fișierul `role_XX.yaml`;
- `[STIMULUS]` este textul nou la care agentul trebuie să reacționeze;
- `[COMENTARII SIMILARE]` sunt fragmentele recuperate din bula lui;
- `prompt` combină rolul, inputul și contextul într-un singur mesaj pentru LLM.
Verificare rapidă:
- apare rolul agentului?
- apare textul nou?
- apar fragmentele recuperate?
- regulile spun clar că agentul nu trebuie să copieze comentariile similare?

In [15]:
print("Rol inclus:", role["name"] in prompt)
print("Input inclus:", input_text in prompt)
print("Context inclus:", retrieved_context[:50] in prompt)

Rol inclus: False
Input inclus: True
Context inclus: True


## 7. Apelăm LLM-ul și generăm răspunsul
Acum trimitem promptul către model.
Acesta este primul răspuns RAG al agentului: răspunsul nu vine doar din model, ci din combinația dintre rol, input și fragmentele recuperate.
Folosim o temperatură mică (`temperature=0.3`) pentru răspunsuri mai stabile și mai ușor de comparat.

In [16]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

MODEL_NAME_LLM = "gemini-2.5-flash-lite"

In [17]:
response = client.chat.completions.create(
    model=MODEL_NAME_LLM,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.3
)

agent_response = response.choices[0].message.content

print(agent_response)


A renunța la Uniunea Europeană ar însemna să ne întoarcem la izolare și să negăm progresul democratic și economic pe care l-am realizat împreună, iar astfel de idei populiste ignoră complet beneficiile concrete ale apartenenței noastre la un spațiu de stabilitate și prosperitate. Instituțiile UE, prin transparența și mecanismele lor juridice, sunt fundamentale pentru consolidarea statului de drept și pentru a ne apăra de manipulare.


- `agent_response` păstrează răspunsul generat de model.


### Verificare manuală
Citește răspunsul generat și completează evaluarea de mai jos.

In [18]:
context_used = "yes"      # yes / partial / no
voice_coherent = "yes"    # yes / partial / no
invented_info = "no"      # yes / unclear / no

notes = "Răspunsul folosește contextul recuperat și păstrează vocea agentului."

print("Folosește contextul:", context_used)
print("Păstrează vocea:", voice_coherent)
print("Inventează informații:", invented_info)
print("Observații:", notes)

Folosește contextul: yes
Păstrează vocea: yes
Inventează informații: no
Observații: Răspunsul folosește contextul recuperat și păstrează vocea agentului.


Întrebări pentru verificare:
- Răspunsul folosește idei sau formulări inspirate din fragmentele recuperate?
- Răspunsul păstrează vocea agentului ales?
- Răspunsul introduce informații care nu apar în input sau în context?
- Răspunsul respectă regula: un singur comentariu, maximum 3 propoziții?

## 8. Același lucru cu LangChain minimal
Până acum am construit promptul manual, cu un `f-string`.
Acum facem același lucru cu LangChain, folosind `PromptTemplate`.
LangChain nu face modelul mai inteligent. Ne ajută să standardizăm promptul și să refolosim aceeași structură pentru mai mulți agenți.
În C6 folosim doar partea minimă:
```text
rol + input + context → șablon de prompt → LLM → răspuns


Nu folosim încă:
- LangGraph
- memorie conversațională
- tools
- agenți complecși
- RetrievalQA


In [19]:
from langchain_core.prompts import PromptTemplate

In [20]:
template = PromptTemplate.from_template("""
{agent_system}

[STIMULUS]
{input_text}

[COMENTARII SIMILARE]
{retrieved_context}
""")
langchain_prompt = template.format(
    agent_system=role["system"],
    input_text=input_text,
    retrieved_context=retrieved_context
)
print(langchain_prompt)


Ești un un cetățean românn pro-european, care conștientizează avantajele apartenenței la Uniunea Europeană, care apără instituțiile UE.
Crezi că instituțiile UE sunt profund necesare, lupți pentru democrație, meritocrație și transparență.
Cum vorbești:
- echilibrat, entuziasmat, didactic
- cu claritate juridică, clar
- argumente contra extremismului, izolaționismului și manipulării
Ce te definește:
- nu ai încredere în extremism, izolaționism
- susții Uniunea Europeană și instituțiile sale
- susții statul de drept și ancorarea europeană
- faci acuza împotriva populismului
-ești pro-european în primul rand nu anti-suveranist
Vei primi:
[STIMULUS] — știrea sau textul la care reacționezi
[COMENTARII SIMILARE] — exemple reale din corpus, utile pentru ton și stil
Reguli:
- scrii ca un comentariu autentic de YouTube în limba română
- folosești comentariile similare doar ca inspirație de ton, nu le copia
- nu explica ce faci
- nu face liste
- nu folosi ghilimele
- răspunde cu un singur comen

Ce face codul:
- `PromptTemplate.from_template()` definește un șablon reutilizabil.
- `{agent_system}`, `{input_text}` și `{retrieved_context}` sunt variabile.
- `.format(...)` completează șablonul cu valorile concrete.
- Rezultatul este un prompt final, la fel ca în varianta manuală.
Diferența importantă: acum structura promptului este standardizată și poate fi refolosită pentru orice agent.

#### Acum trimitem promptul construit cu LangChain către același model.

In [21]:
response_lc = client.chat.completions.create(
    model=MODEL_NAME_LLM,
    messages=[
        {
            "role": "user",
            "content": langchain_prompt
        }
    ],
    temperature=0.3
)
agent_response_lc = response_lc.choices[0].message.content
print(agent_response_lc)

A afirma că România ar trebui să părăsească Uniunea Europeană este o abordare periculoasă, care ignoră beneficiile imense aduse de apartenența noastră la acest proiect european, un proiect bazat pe democrație, stat de drept și prosperitate. A ne izola acum, când avem posibilitatea de a contribui activ la consolidarea valorilor europene și de a beneficia de stabilitate, ar fi o greșeală istorică, alimentată de populism și manipulare.


### Mini-task
Schimbă doar `input_text`, apoi rulează din nou pașii de retrieval, construire context și prompt.
Observă că șablonul rămâne același. Se schimbă doar datele introduse în el.
LangChain este util aici pentru că separă clar:
```text
structura promptului
de
valorile concrete: rol, input, context

## 9. Testăm două inputuri
Nu vrem să testăm agentul pe un singur exemplu. Un agent RAG trebuie verificat pe mai multe inputuri, ca să vedem dacă păstrează vocea și dacă folosește contextul recuperat.
În acest pas rulăm același agent pe două texte politice scurte.

In [22]:
test_inputs = [
    "CCR a decis anularea alegerilor după suspiciuni privind influențe externe.",
    "Guvernul a anunțat noi măsuri economice care au provocat proteste."
]

In [23]:
def retrieve_context(input_text, k=5):
    query_embedding = model.encode(
        [input_text],
        normalize_embeddings=True
    ).astype("float32")

    scores, positions = index.search(query_embedding, k)

    results = []
    for score, pos in zip(scores[0], positions[0]):
        item = metadata[pos].copy()
        item["score"] = round(float(score), 3)
        results.append(item)

    context_parts = []
    for i, item in enumerate(results, start=1):
        context_parts.append(
            f"""[Fragment {i} | score={item.get("score", "")} | source={item.get("source_channel", "")}]
{item.get("text", "")}
"""
        )

    return results, "\n".join(context_parts)

In [24]:
def generate_response(input_text):
    results, retrieved_context = retrieve_context(input_text, k=K)

    final_prompt = template.format(
        agent_system=role["system"],
        input_text=input_text,
        retrieved_context=retrieved_context
    )

    response = client.chat.completions.create(
        model=MODEL_NAME_LLM,
        messages=[{"role": "user", "content": final_prompt}],
        temperature=0.3
    )

    return {
        "agent_slug": MY_AGENT,
        "agent_name": role["name"],
        "input_text": input_text,
        "retrieved_context": results,
        "prompt": final_prompt,
        "response": response.choices[0].message.content,
        "model": MODEL_NAME_LLM,
        "temperature": 0.3
    }

In [25]:
test_results = []

for text in test_inputs:
    result = generate_response(text)
    test_results.append(result)

    print("=" * 80)
    print("INPUT:")
    print(result["input_text"])
    print("\nRĂSPUNS:")
    print(result["response"])

INPUT:
CCR a decis anularea alegerilor după suspiciuni privind influențe externe.

RĂSPUNS:
Este esențial să apărăm integritatea proceselor democratice, iar deciziile precum cea a CCR, bazate pe suspiciuni de influențe externe, subliniază importanța vigilenței noastre comune împotriva oricăror tentative de subminare a statului de drept. Aceste momente ne reamintesc de ce ancorarea europeană și instituțiile UE sunt fundamentale pentru a asigura un cadru stabil și transparent pentru democrația noastră.
INPUT:
Guvernul a anunțat noi măsuri economice care au provocat proteste.

RĂSPUNS:
Măsurile economice anunțate de guvern, care au stârnit proteste, necesită o analiză atentă prin prisma principiilor democratice și a statului de drept, pentru a ne asigura că sunt echilibrate și transparente. Este esențial să evităm soluțiile populiste sau izolaționiste și să ne concentrăm pe consolidarea apartenenței noastre europene, care oferă stabilitate și prosperitate.


context_used: yes 

voice_coherent: yes 

problems: none

# 9. Mini-agent RAG cu tool de regăsire
 
Până acum:
noi am făcut retrieval manual → am pus contextul în prompt → am apelat LLM-ul.
 
Acum:
definim retrieval-ul ca tool → agentul poate folosi tool-ul → apoi generează răspunsul.

In [26]:
!pip install langchain-openai

In [27]:
!pip install langchain

In [28]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

In [29]:
PROVIDER = "deepseek"  # "gemini" sau "deepseek"
if PROVIDER == "gemini":
    MODEL_NAME_AGENT = "gemini-2.5-flash-lite"
    API_KEY = "GEMINI_API_KEY"
    BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
elif PROVIDER == "deepseek":
    MODEL_NAME_AGENT = "deepseek-chat"
    API_KEY = os.getenv("DEEPSEEK_API_KEY")
    BASE_URL = "https://api.deepseek.com/v1"
else:
    raise ValueError("Provider necunoscut. Alege 'gemini' sau 'deepseek'.")
 
llm = ChatOpenAI(
    model=MODEL_NAME_AGENT,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0.3,
)
print("Provider:", PROVIDER)
print("Model:", MODEL_NAME_AGENT)

Provider: deepseek
Model: deepseek-chat


In [30]:
@tool
def retrieve_similar_comments(query: str) -> str:
    """Caută comentarii similare în bula discursivă a agentului."""
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")
   
    scores, positions = index.search(query_embedding, K)
    context_parts = []
    for i, (score, pos) in enumerate(zip(scores[0], positions[0]), start=1):
        item = metadata[pos]
        context_parts.append(
            f"""
[Fragment {i} | score={round(float(score), 3)}]
{item.get("text", "")}
"""
        )
    return "\n".join(context_parts)

In [31]:
agent = create_agent(
    model=llm,
    tools=[retrieve_similar_comments],
    system_prompt=role["system"] + """
 
    REGULĂ OBLIGATORIE:
    Înainte să răspunzi, trebuie să folosești instrumentul `retrieve_similar_comments`
    pentru a căuta comentarii similare în corpusul agentului.
 
    Nu răspunde direct fără să folosești instrumentul.
 
După ce primești comentariile similare:
- folosește-le doar ca inspirație de ton și stil;
- nu le copia;
- răspunde cu un singur comentariu;
- maximum 3 propoziții.
"""
)

In [32]:
input_text = "CCR a decis anularea alegerilor după suspiciuni privind influențe externe."
agent_result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": input_text
        }
    ]
})
print(agent_result["messages"][-1].content)

O decizie grea, dar necesară pentru a proteja democrația și statul de drept în fața unor ingerințe străine care au încercat să deturneze voința alegătorilor. Instituțiile UE și cele naționale trebuie să colaboreze mai strâns ca niciodată pentru a preveni astfel de atacuri hibride la adresa proceselor electorale. Susțin pe deplin acest gest de curaj al magistraților care au pus mai presus legea și securitatea națională decât populismul ieftin.


In [33]:
# ne uitam daca a folosit tool
for message in agent_result["messages"]:
    print(type(message).__name__)
    print(message)
    print("-" * 80)

HumanMessage
content='CCR a decis anularea alegerilor după suspiciuni privind influențe externe.' additional_kwargs={} response_metadata={} id='5df291dd-1b01-405b-bd64-4813c1bc0d55'
--------------------------------------------------------------------------------
AIMessage
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 797, 'total_tokens': 862, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 768}, 'prompt_cache_hit_tokens': 768, 'prompt_cache_miss_tokens': 29}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '5a5e11cb-b988-4abb-af3f-62459c16c6a0', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019e2fa5-7b8c-7ca2-8360-de142bf89cdf-0' tool_calls=[{'name': 'retrieve_similar_comments', 'args': {'query': 'CCR anulare alegeri influențe externe decizie justiție'}, 

In [34]:
used_tool = any(
    hasattr(message, "tool_calls") and len(message.tool_calls) > 0
    for message in agent_result["messages"]
)
print("Agentul a folosit tool-ul:", used_tool)

Agentul a folosit tool-ul: True


## 10. Mini-agent RSS: de la știre recentă la comentariu de bulă
Până acum am dat noi manual un text politic agentului.
Acum facem un pas mai agentic: agentul primește acces la două instrumente:
1. un instrument care citește o știre recentă dintr-un feed RSS;
2. un instrument care caută comentarii similare în bula discursivă a agentului.
Fluxul devine:
```text
RSS news → retrieve similar comments → role_XX.yaml → LLM → comentariu de bulă

In [35]:
%pip install -U feedparser

Note: you may need to restart the kernel to use updated packages.


In [36]:
import feedparser
from langchain_core.tools import tool

### 10.2 Alegem o sursă RSS
Pentru laborator folosim o sursă RSS publică. Poți schimba feed-ul dacă vrei să testezi altă sursă.
Exemple posibile:
 
https://www.g4media.ro/feed
 
https://www.hotnews.ro/rss

In [37]:
RSS_FEED = "https://www.hotnews.ro/rss"

In [38]:
@tool
def get_latest_news_from_rss() -> str:
    """Ia cea mai recentă știre din feed-ul RSS și returnează titlul, linkul și rezumatul."""
    feed = feedparser.parse(RSS_FEED)
   
    if not feed.entries:
        return "Nu am găsit știri în feed-ul RSS."
   
    entry = feed.entries[0]
   
    title = entry.get("title", "")
    link = entry.get("link", "")
    summary = entry.get("summary", "")
   
    return f"""
TITLU:
{title}
 
LINK:
{link}
 
REZUMAT:
{summary}
"""
 
 
import feedparser
 
RSS_FEED = "https://www.g4media.ro/feed"
 
feed = feedparser.parse(RSS_FEED)
 
print("Număr știri:", len(feed.entries))
feed.entries[0]
 

Număr știri: 10


{'title': 'Ploaia a oprit a doua semifinală de la Roma: Sinner – Medvedev se reia astăzi',
 'title_detail': {'type': 'text/plain',
  'language': None,
  'base': 'https://www.g4media.ro/feed',
  'value': 'Ploaia a oprit a doua semifinală de la Roma: Sinner – Medvedev se reia astăzi'},
 'links': [{'rel': 'alternate',
   'type': 'text/html',
   'href': 'https://www.g4media.ro/ploaia-a-oprit-a-doua-semifinala-de-la-roma-sinner-medvedev-se-reia-astazi.html'},
  {'length': '500',
   'type': 'image/jpeg',
   'href': 'https://www.g4media.ro//wp-content/uploads/2026/04/8116002-Mediafax_Foto-DPA_Hepta-1024x714.jpg',
   'rel': 'enclosure'}],
 'link': 'https://www.g4media.ro/ploaia-a-oprit-a-doua-semifinala-de-la-roma-sinner-medvedev-se-reia-astazi.html',
 'comments': 'https://www.g4media.ro/ploaia-a-oprit-a-doua-semifinala-de-la-roma-sinner-medvedev-se-reia-astazi.html#respond',
 'authors': [{'name': 'Andrei Ciobanu'}],
 'author': 'Andrei Ciobanu',
 'author_detail': {'name': 'Andrei Ciobanu'},
 '

In [39]:
# Testăm tool-ul RSS înainte să îl dăm agentului
latest_news = get_latest_news_from_rss.invoke({})
print(latest_news)


TITLU:
Ploaia a oprit a doua semifinală de la Roma: Sinner – Medvedev se reia astăzi

LINK:
https://www.g4media.ro/ploaia-a-oprit-a-doua-semifinala-de-la-roma-sinner-medvedev-se-reia-astazi.html

REZUMAT:
<p>Duelul dintre Jannik Sinner și Daniil Medvedev din semifinalele Mastersului de la Roma a fost întrerupt de ploaie vineri seară, iar finalistul din această întâlnire va fi dat astăzi, 16 mai. Partida a fost întreruptă în timp ce Sinner (1 ATP) și Medvedev (9 ATP) se luptau în setul decisiv la Foro Italico (scor 6-2, [&#8230;]</p>
<p>&copy; <a href="https://www.g4media.ro">G4Media.ro</a>.</p>



### TODO — explică ce face tool-ul RSS
Completează:
- `feedparser.parse(RSS_FEED)` face: preia valorile din RSS_FEED.
- `feed.entries[0]` selectează: prima valoarea
- Tool-ul returnează trei informații: titlu, link, rezumat
- De ce este util să testăm tool-ul înainte să îl dăm agentului? ca sa vedem daca functioneaza

In [40]:
feed = feedparser.parse(RSS_FEED)
 
print("Feed title:", feed.feed.get("title", ""))
print("Număr știri găsite:", len(feed.entries))
 
entry = feed.entries[0]
print("Titlu:", entry.get("title", ""))
print("Link:", entry.get("link", ""))

Feed title: G4Media.ro
Număr știri găsite: 10
Titlu: Ploaia a oprit a doua semifinală de la Roma: Sinner – Medvedev se reia astăzi
Link: https://www.g4media.ro/ploaia-a-oprit-a-doua-semifinala-de-la-roma-sinner-medvedev-se-reia-astazi.html


### 10.4 Tool 2: căutăm comentarii similare în bula agentului
Acest tool reutilizează mecanismul FAISS construit în C5.
Diferența este că acum îl ambalăm ca tool pentru agent.

In [41]:
@tool
def retrieve_similar_comments(query: str) -> str:
    """Caută comentarii similare în bula discursivă a agentului."""
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")
   
    scores, positions = index.search(query_embedding, K)
   
    context_parts = []
   
    for i, (score, pos) in enumerate(zip(scores[0], positions[0]), start=1):
        item = metadata[pos]
        fragment = f"""
[Comentariu similar {i} | score={round(float(score), 3)}]
{item.get("text", "")}
"""
        context_parts.append(fragment)
   
    return "\n".join(context_parts)
 
# Testăm tool-ul FAISS separat
test_query = "CCR a decis anularea alegerilor după suspiciuni privind influențe externe."
similar_comments = retrieve_similar_comments.invoke({"query": test_query})
print(similar_comments)
 


[Comentariu similar 1 | score=0.364]
Întrebări punctuale și ar fi bine să rămână în viitor pentru a fi ridicate la fileu atunci când este cazul pentru a fi rezolvate din timp pentru situații similare, așa cum a fost cu alegerile comasate, și anularea turului doi prezidențial,din CCR și structurile din serviciile de informații.


[Comentariu similar 2 | score=0.319]
@Robert Turcescu Oficial: întrebare pentru dnul Călin Georgescu: ce soluție recomandă dânsul pentru ieșirea României din criza în care se află, și întoarcerea la statul de drept și la Constituție? Ce părere are despre inițiativa civică VALUL DEMOCRAȚIEI care a depus până acum peste 400 de plângeri penale la Parchet, plângeri pe care Parchetul General refuză să le instrumenteze și să le comaseze într-un dosar penal? Cum explică dânsul faptul că unii susținători ai dânsului, vizibili la Buftea, atacă inițiativa Valul Democrației și îndeamnă oamenii să nu depună plângerile penale?


[Comentariu similar 3 | score=0.26]
Ce sunt 

### TODO — explică tool-ul de regăsire
Completează:
- Acest tool primește ca input: un text / o întrebare nouă
- Transformă inputul în: embedding vectorial
- Caută în: vectorstore / baza de comentarii similare
- Returnează: comentarii similare cu scor de similaritate
- De ce acest tool este diferit de simpla generare cu LLM? pentru că recuperează context real din date înainte să genereze răspunsul

### 10.5 Creăm agentul cu două instrumente
Agentul are acum:
- rolul discursiv din `role_XX.yaml`;
- un tool pentru știri recente;
- un tool pentru comentarii similare.
Instrucțiunea importantă: agentul trebuie să folosească mai întâi RSS-ul, apoi regăsirea semantică.

In [42]:
agent_news = create_agent(
    model=llm,
    tools=[get_latest_news_from_rss, retrieve_similar_comments],
    system_prompt=role["system"] + """
 
Ai două instrumente:
1. get_latest_news_from_rss — citește o știre recentă dintr-un feed RSS.
2. retrieve_similar_comments — caută comentarii similare în bula discursivă.
 
REGULĂ OBLIGATORIE:
Folosește mai întâi get_latest_news_from_rss.
Apoi folosește retrieve_similar_comments pe titlul sau rezumatul știrii.
 
După ce ai primit ambele rezultate, scrie:
 
ȘTIRE FOLOSITĂ:
titlul știrii și linkul
 
COMENTARIU:
un singur comentariu de YouTube, maximum 3 propoziții, în vocea agentului
 
NOTĂ:
o propoziție scurtă despre ce a venit din știre și ce a venit din bula discursivă.
 
Nu prezenta interpretarea agentului ca fapt verificat.
"""
)

### 10.6 Rulăm mini-agentul RSS
Acum nu mai scriem noi inputul politic.
Îi cerem agentului să ia o știre recentă și să o comenteze.

In [43]:
agent_news_result = agent_news.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Alege o știre recentă din RSS și comenteaz-o în vocea agentului."
        }
    ]
})
 
print(agent_news_result["messages"][-1].content)

ȘTIRE FOLOSITĂ:
Ploaia a oprit a doua semifinală de la Roma: Sinner – Medvedev se reia astăzi
https://www.g4media.ro/ploaia-a-oprit-a-doua-semifinala-de-la-roma-sinner-medvedev-se-reia-astazi.html

COMENTARIU:
Mă bucur să văd că sportul de performanță și valorile fair-play-ului rămân în prim-plan, exact cum ar trebui să fie și în politică și justiție. În timp ce un meci se reia după o întrerupere naturală, în România asistăm la întreruperi artificiale ale statului de drept din cauza populismului toxic. Susțin valorile europene și transparența, nu haosul și manipularea.

Din știre am preluat contextul sportiv, iar din comentariile similare am extras tonul civic și apelul la statul de drept și democrație.


### 10.7 Verificăm dacă agentul a folosit instrumentele
Un agent cu tool-uri trebuie verificat.
Nu este suficient să vedem răspunsul final. Trebuie să vedem dacă a apelat instrumentele.

In [44]:
for message in agent_news_result["messages"]:
    print(type(message).__name__)
   
    if hasattr(message, "tool_calls"):
        print("tool_calls:", message.tool_calls)
   
    print(str(message.content)[:1200])
    print("-" * 80)

HumanMessage
Alege o știre recentă din RSS și comenteaz-o în vocea agentului.
--------------------------------------------------------------------------------
AIMessage
tool_calls: [{'name': 'get_latest_news_from_rss', 'args': {}, 'id': 'call_00_ahnSsO43L5CS4boEnk976125', 'type': 'tool_call'}]
Am să încep prin a prelua cea mai recentă știre din RSS.
--------------------------------------------------------------------------------
ToolMessage

TITLU:
Ploaia a oprit a doua semifinală de la Roma: Sinner – Medvedev se reia astăzi

LINK:
https://www.g4media.ro/ploaia-a-oprit-a-doua-semifinala-de-la-roma-sinner-medvedev-se-reia-astazi.html

REZUMAT:
<p>Duelul dintre Jannik Sinner și Daniil Medvedev din semifinalele Mastersului de la Roma a fost întrerupt de ploaie vineri seară, iar finalistul din această întâlnire va fi dat astăzi, 16 mai. Partida a fost întreruptă în timp ce Sinner (1 ATP) și Medvedev (9 ATP) se luptau în setul decisiv la Foro Italico (scor 6-2, [&#8230;]</p>
<p>&copy; <a hr

In [45]:
used_tools = []
 
for message in agent_news_result["messages"]:
    if hasattr(message, "tool_calls"):
        for call in message.tool_calls:
            used_tools.append(call["name"])
 
print("Tool-uri folosite:", used_tools)
print("A folosit RSS:", "get_latest_news_from_rss" in used_tools)
print("A folosit FAISS:", "retrieve_similar_comments" in used_tools)

Tool-uri folosite: ['get_latest_news_from_rss', 'retrieve_similar_comments']
A folosit RSS: True
A folosit FAISS: True


### TODO — concluzie scurtă
Scrie 3–4 fraze:
1. Ce a făcut agentul diferit față de varianta manuală? (aici putem răspunde doar in gând)
1. Ce ar trebui verificat de un om înainte ca acest răspuns să fie folosit într-o aplicație publică?


In [46]:
from pathlib import Path
import yaml

path = Path("assets/roles/roles.yaml")

try:
    with open(path, "r", encoding="utf-8") as f:
        roles = yaml.safe_load(f)

    print(roles.keys())
    print(roles["agents"].keys())

except yaml.YAMLError as e:
    print(e)

dict_keys(['agents'])
dict_keys(['anti_suveranist', 'conspirationist', 'pro_european'])


In [47]:
from pathlib import Path
import yaml

path = Path("assets/roles/roles.yaml")

try:
    with open(path, "r", encoding="utf-8") as f:
        roles = yaml.safe_load(f)

    print(roles.keys())
    print(roles["agents"].keys())

except yaml.YAMLError as e:
    print(e)

dict_keys(['agents'])
dict_keys(['anti_suveranist', 'conspirationist', 'pro_european'])
